# ColQwen2.5 Visual Retrieval — Colab L4 (Phase 5 · VISUAL-01 / VISUAL-02)

This notebook produces the **real** visual-retrieval numbers (recall@5/@10, citation accuracy)
and proves whether the Example-3 image-only pages become retrievable. Nothing here is mocked — it's the
no-fabrication boundary for the phase.

---

## ▶ How to run — just 4 steps (use **Run all**, never run cells one by one)

1. **Runtime → Change runtime type → L4 GPU.**
2. **Runtime → Run all.**
3. **If a grey `RESTART SESSION` button appears under the install cell (§1)** — click it, then
   **Runtime → Run all** again. That's expected once: the install swaps Colab's torch/transformers
   for the versions ColQwen needs, and the restart activates them. Run-all then re-runs everything
   **in the correct order**, so you never hit "No module named src" (that error only happens when
   cells are run out of order by hand).
4. When the **"Upload compliance.db"** cell asks, upload your local **`compliance.db`** (the one
   with page images — it's gitignored and never committed).

Then read the **load report**, **metrics table**, **Example-3 proof**, and the exported run report.

---

## Repo access — do this once before running

This notebook clones the repo to import the shared `src/` code.

- **Easiest (recommended): make the repo PUBLIC.** It's source code only — your PDFs and
  `compliance.db` are gitignored, so nothing sensitive is exposed. With a public repo the clone
  needs **no token** and there's nothing to leak.
- **Private repo instead?** Add a Colab Secret (🔑 left sidebar) named exactly **`GH_TOKEN`** =
  a fine-grained PAT with **Contents: Read** granted to *this* repo. The §2 cell uses it and
  **never prints it** (failures are redacted).

No API keys are needed — retrieval is provider-free.

---

*Primary stack: `colpali-engine==0.3.9`, `transformers>=4.50,<4.51`, `torch==2.6.0`,
`torchvision==0.21.0`, `peft>=0.14,<0.15`, `accelerate>=0.34,<1.4`.*

*Fallback experiment: upstream `colpali-engine` main at commit `c23838d920a7c426ee297034211cff2f55da65dc`
with Transformers 5.x. Use it only if the primary 4.x stack cannot resolve or load cleanly.*

## 1. Install the primary fixed stack

Primary fix = **Option B**: use the earliest PyPI ColQwen2.5-capable pre-Transformers-5 stack.
This avoids the `model.*` → `language_model.*` checkpoint conversion path entirely, so the
`vidore/colqwen2.5-v0.2` base + LoRA checkpoint layout matches the runtime model layout.

Pinned set:

- `colpali-engine==0.3.9`
- `transformers>=4.50,<4.51`
- `torch==2.6.0` + `torchvision==0.21.0`
- `peft>=0.14,<0.15`
- `accelerate>=0.34,<1.4`

The cell uninstalls Colab's preinstalled `torchao` first; it is not used for bf16 retrieval and has
caused resolver/import conflicts in previous runs. Restart the runtime afterward if Colab prompts you.

Fallback Option A (documented, not default): install upstream main at commit
`c23838d920a7c426ee297034211cff2f55da65dc`, which adds the missing
`model.embed_tokens`/`model.norm` remaps for Transformers 5.x. The source also maps
LoRA-shaped `model.layers.*.lora_*` keys to `language_model.layers.*.lora_*`, but because 0.3.17
already had that textual rule while the live adapter load still failed, this remains the fallback
until the load report proves it clean.

In [ ]:
!pip uninstall -q -y torchao
!pip install -q --upgrade --force-reinstall "numpy<2.1" "pillow<12" "torch==2.6.0" "torchvision==0.21.0" "colpali-engine==0.3.9" "transformers>=4.50,<4.51" "peft>=0.14,<0.15" "accelerate>=0.34,<1.4" "qdrant-client>=1.17,<2.0" pypdfium2 loguru "pydantic>=2.8" pydantic-settings python-dateutil tenacity psutil

# Fallback Option A (do not run unless the primary stack cannot resolve/load):
# !pip uninstall -q -y torchao
# !pip install -q --upgrade --force-reinstall "git+https://github.com/illuin-tech/colpali.git@c23838d920a7c426ee297034211cff2f55da65dc" "transformers>=5.3,<6" "torch>=2.2,<2.12" "peft>=0.18,<0.20" "qdrant-client>=1.17,<2.0" pypdfium2 pillow loguru "pydantic>=2.8" pydantic-settings python-dateutil tenacity psutil

## ⚠ If a `RESTART SESSION` button appears after the install above

Click it, then **Runtime → Run all** again. This happens at most once (the install swapped torch/transformers). Run-all then re-runs every cell **in order**, so the "No module named src" error cannot happen — you never run cells by hand.

## 2. Set up — clone the repo and put `src/` on the path

Run-all reaches this right after the install. With a **public** repo this needs no token; with a private repo it uses the `GH_TOKEN` Colab secret (and never prints it).

In [ ]:
import os, sys, subprocess

# RECOMMENDED: make this repo PUBLIC — then no token is needed and nothing can leak.
# Private alternative: set a Colab secret GH_TOKEN (fine-grained PAT, Contents:Read on THIS repo).
REPO_URL = os.environ.get("REPO_URL", "https://github.com/aatif101/pfizer-externship.git")
REPO_DIR = "/content/pfizer-externship"

token = None
try:
    from google.colab import userdata  # type: ignore
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None  # public repo, or src/ uploaded under /content

clone_url = REPO_URL
if token and clone_url.startswith("https://github.com/"):
    clone_url = clone_url.replace("https://github.com/", f"https://{token}@github.com/")


def _git(args):
    res = subprocess.run(args, capture_output=True, text=True)
    if res.returncode != 0:
        stderr = (res.stderr or "").replace(token or "\0", "***")  # never echo the token
        raise RuntimeError(
            f"git failed (exit {res.returncode}):\n{stderr}\n"
            "Fix: make the repo PUBLIC (simplest), or set a GH_TOKEN secret with "
            "Contents:Read on THIS repo."
        )


if not os.path.isdir(REPO_DIR):
    _git(["git", "clone", "--depth", "1", clone_url, REPO_DIR])
else:
    # Already cloned — pull the latest main so reruns always pick up fixes.
    _git(["git", "-C", REPO_DIR, "fetch", "--depth", "1", clone_url, "main"])
    _git(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"])

if os.path.isdir(REPO_DIR):
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
elif os.path.isdir("/content/src"):
    if "/content" not in sys.path:
        sys.path.insert(0, "/content")  # fallback: src/ uploaded directly under /content

import src.retrieval.visual.collection  # noqa: F401  fail fast if the path is wrong
print("src.retrieval.visual importable:", True)

## 3. Version print + load-report gate + smoke embed

Print the exact `(colpali-engine, transformers, torch, peft)` tuple that loaded, then load the model
via the shared `embedder.load_colqwen` seam with `output_loading_info=True`.

**Hard gate:** the load report must show no critical missing/unexpected/mismatched keys for
`embed_tokens`, `norm`, or `lora_*`. The previous broken stack passed shape checks while dropping
these weights, so this cell fails before any indexing if the model is silently corrupted.

For this primary 4.x stack, `STRICT_ZERO_LOAD_MISSING=True` also requires zero total missing keys.
If a future upstream release introduces a known-harmless missing key, keep the critical-key guard and
record the exception explicitly in the run report instead of weakening the gate silently.

In [ ]:
import gc
import importlib.metadata as _md
import json
import colpali_engine  # noqa: F401  (import proves it loaded; version comes from metadata)
import transformers
import torch
from PIL import Image


def _ver(pkg: str) -> str:
    # colpali_engine has no __version__ attribute — read the installed dist version.
    try:
        return _md.version(pkg)
    except Exception:
        return "unknown"


VERSION_TRIPLE = {
    "colpali_engine": _ver("colpali-engine"),
    "transformers": _ver("transformers"),
    "torch": _ver("torch"),
    "peft": _ver("peft"),
}
print("version triple:", json.dumps(VERSION_TRIPLE, indent=2))
print("cuda available :", torch.cuda.is_available())
assert torch.cuda.is_available(), "This notebook requires a GPU (Colab L4). Runtime -> Change runtime type -> L4 GPU."
print("gpu            :", torch.cuda.get_device_name(0))

from src.retrieval.visual.embedder import (
    assert_colqwen_load_report_clean,
    embed_images,
    embed_queries,
    load_colqwen,
    pooled_vectors_for_image,
    process_image_batch,
)

model, processor, LOAD_REPORT = load_colqwen(return_load_report=True)  # vidore/colqwen2.5-v0.2, bf16, cuda:0
print("LOAD REPORT:", json.dumps(LOAD_REPORT, indent=2))
assert_colqwen_load_report_clean(LOAD_REPORT)
STRICT_ZERO_LOAD_MISSING = True
if STRICT_ZERO_LOAD_MISSING:
    assert LOAD_REPORT["missing_count"] == 0, f"Expected zero missing keys, got {LOAD_REPORT['missing_keys'][:20]}"
    assert LOAD_REPORT["mismatched_count"] == 0, f"Expected zero mismatched keys, got {LOAD_REPORT['mismatched_keys'][:20]}"
# Unexpected adapter/base keys are also suspicious; keep this strict for the ColQwen regression gate.
assert LOAD_REPORT["unexpected_count"] == 0, f"Expected zero unexpected keys, got {LOAD_REPORT['unexpected_keys'][:20]}"

# RESEARCH A1/A2: confirm dim==128 and inspect the dynamic-grid attributes BEFORE the full build.
smoke_img = Image.new("RGB", (768, 1024), color="white")
smoke_emb = embed_images(model, processor, [smoke_img])
print("smoke image embedding shape:", tuple(smoke_emb.shape), "-> per-token dim =", int(smoke_emb.shape[-1]))
assert smoke_emb.shape[-1] == 128, f"expected per-token dim 128, got {smoke_emb.shape[-1]} (update collection size if this fires)"
print("model.patch_size =", getattr(model, "patch_size", None), " spatial_merge_size =", getattr(model, "spatial_merge_size", None))
del smoke_emb; torch.cuda.empty_cache(); gc.collect()

## 4. Upload `compliance.db` and read pages from `pages.image_blob`

`compliance.db` carries the per-page `image_blob` (the reproducible 150-DPI PNG). We rasterize
from the stored blob via the shared `pooling.blob_to_image` — **never** from the gitignored
source PDFs. Only page identities + image bytes leave the DB; no full page text is dumped to output.

In [ ]:
import sqlite3

DB_PATH = "/content/compliance.db"
if not os.path.exists(DB_PATH):
    from google.colab import files  # type: ignore
    print("Upload compliance.db (with pages.image_blob) ...")
    uploaded = files.upload()
    name = next(iter(uploaded))
    if name != "compliance.db":
        os.replace(name, DB_PATH)
assert os.path.exists(DB_PATH), "compliance.db not found at /content/compliance.db"

from src.retrieval.visual.pooling import blob_to_image

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
# Parameterized-shape SQL (no string interpolation): every page incl. image-only ones.
rows = conn.execute(
    "SELECT p.doc_id AS doc_id, p.page_num AS page_num, d.filename AS filename, "
    "p.image_blob AS image_blob, LENGTH(TRIM(COALESCE(p.page_text, ''))) AS page_text_len "
    "FROM pages p JOIN documents d ON d.doc_id = p.doc_id "
    "WHERE p.image_blob IS NOT NULL "
    "ORDER BY p.doc_id, p.page_num"
).fetchall()
conn.close()

pages = [
    {
        "doc_id": r["doc_id"],
        "page_num": int(r["page_num"]),  # 0-indexed throughout (RESEARCH Pitfall 6)
        "filename": r["filename"],
        "page_text_len": int(r["page_text_len"]),
        "image": blob_to_image(bytes(r["image_blob"])),
    }
    for r in rows
]
PAGE_TEXT_LENGTHS = {(p["doc_id"], p["page_num"]): p["page_text_len"] for p in pages}
print(f"pages to embed: {len(pages)}  across {len({p['doc_id'] for p in pages})} documents")
print("original empty-text pages:", sum(1 for length in PAGE_TEXT_LENGTHS.values() if length == 0))

## 5. Build the `sdf_page_images` Qdrant collection (3 named vectors)

Uses the shared `collection.build_vectors_config()` (original HNSW-off + mean_pooling_rows/
columns) and `collection.build_upsert_point(...)`. Embedding runs in bf16 at `batch_size=2`
with `empty_cache()+gc.collect()` between batches (RESEARCH Pitfall 3). For each page we derive
the row/column pooled multivectors via the shared `embedder.pooled_vectors_for_image`
(which delegates the reshape to the offline-tested `pooling.mean_pool_rows_cols`). Payload
carries 0-indexed `page_num` + `doc_id` ONLY — no image bytes, no page text.

In [ ]:
from qdrant_client import QdrantClient
from src.retrieval.visual.collection import build_vectors_config, collection_name, build_upsert_point

VERSION = 1
BATCH_SIZE = 2
QDRANT_PATH = "/content/qdrant_storage"

client = QdrantClient(path=QDRANT_PATH)
name = collection_name(VERSION)
if client.collection_exists(name):
    client.delete_collection(name)
client.create_collection(collection_name=name, vectors_config=build_vectors_config())

def _to_list(t):
    return t.detach().to(torch.float32).cpu().numpy().tolist()

upserted = 0
for start in range(0, len(pages), BATCH_SIZE):
    batch = pages[start : start + BATCH_SIZE]
    images = [p["image"] for p in batch]
    batch_images = process_image_batch(processor, images).to(model.device)
    with torch.no_grad():
        image_embeddings = model(**batch_images)  # [B, seq, 128]
    points = []
    for i, p in enumerate(batch):
        pooled_rows, pooled_cols = pooled_vectors_for_image(
            model, processor, batch_images, image_embeddings, i, p["image"].size
        )
        points.append(
            build_upsert_point(
                p["doc_id"], p["page_num"],
                _to_list(image_embeddings[i]),  # original full multivector
                _to_list(pooled_rows),
                _to_list(pooled_cols),
            )
        )
    client.upsert(collection_name=name, points=points)
    upserted += len(points)
    del image_embeddings, batch_images
    torch.cuda.empty_cache(); gc.collect()
    print(f"  upserted {upserted}/{len(pages)} pages", end="\r")

info = client.get_collection(name)
print(f"\ncollection {name}: points={client.count(name).count} indexed_vectors_count={info.indexed_vectors_count}")

# Persist a versioned visual run record (counts/run_id/model_version only).
from src.retrieval.visual.run import build_visual_index_run
pages_meta = [(p["doc_id"], p["page_num"], p["filename"]) for p in pages]
visual_run = build_visual_index_run(DB_PATH, pages_meta, model_version="vidore/colqwen2.5-v0.2", version=VERSION)
print("visual run_id:", visual_run.run_id, "| indexed pages:", visual_run.indexed_page_count)

## 6. Two-stage visual retrieval over the gold queries

Embed each repaired gold query, run the canonical two-stage query via the shared
`querier.build_query_payload` (two mean-pooled HNSW prefetches -> `using="original"` MAX_SIM
rerank), and map the response with `querier.map_response_to_candidates` (0-indexed identity).

For diagnostics this requests the full corpus size as `search_limit`, so the report can record the
true target rank instead of only top-10 hit/miss.

In [ ]:
from src.retrieval.visual.querier import build_query_payload, map_response_to_candidates

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
gold_queries = [
    {"query_id": r["query_id"], "query_text": r["query_text"]}
    for r in conn.execute("SELECT query_id, query_text FROM gold_retrieval_queries ORDER BY query_id").fetchall()
]
gold_targets = {}
for q in gold_queries:
    tgt = conn.execute(
        "SELECT doc_id, page_num FROM gold_retrieval_targets WHERE query_id = ? ORDER BY doc_id, page_num",
        (q["query_id"],),
    ).fetchall()
    gold_targets[q["query_id"]] = [(str(t["doc_id"]), int(t["page_num"])) for t in tgt]
conn.close()
print(f"gold queries: {len(gold_queries)}")

TOP_K = 10
VISUAL_SEARCH_LIMIT = len(pages)
visual_hits_by_query = {}
visual_ranked_by_query = {}
for q in gold_queries:
    q_emb = embed_queries(model, processor, [q["query_text"]])
    payload = build_query_payload(_to_list(q_emb[0]), prefetch_limit=200, search_limit=VISUAL_SEARCH_LIMIT, version=VERSION)
    response = client.query_points(**payload)
    candidates = map_response_to_candidates(response.points)
    visual_hits_by_query[q["query_id"]] = [
        {"doc_id": c.doc_id, "page_num": int(c.page_num), "score": float(c.score)} for c in candidates
    ]
    visual_ranked_by_query[q["query_id"]] = [(c.doc_id, int(c.page_num)) for c in candidates]
    del q_emb; torch.cuda.empty_cache(); gc.collect()
print("two-stage retrieval complete for all gold queries")

## 7. Evaluate text-only vs gated visual-fused on the SAME gold set + print the metrics table

Both modes use the UNCHANGED metric functions (`compute_retrieval_recall_at_k`,
`compute_page_level_citation_accuracy`). Text-only uses the existing `retrieve_evidence`
page identities; visual-fused uses the shared text-first weighted/gated `fusion.rrf_fuse`.

Hard acceptance: fused recall@5 and recall@10 must be **greater than or equal to** text-only.
If visual retrieval is noisy, the notebook fails instead of silently degrading the verified text tier.

In [ ]:
from src.retrieval.retriever import retrieve_evidence
from src.retrieval.visual.fusion import assert_fused_recall_not_below_text, rrf_fuse
from src.eval.retrieval_metrics import compute_retrieval_recall_at_k, compute_page_level_citation_accuracy

K_VALUES = (5, 10)

# Text-only ranked page identities (existing path, unchanged).
text_ranked_by_query = {}
for q in gold_queries:
    res = retrieve_evidence(DB_PATH, q["query_text"], top_k=max(K_VALUES))
    text_ranked_by_query[q["query_id"]] = [(h.doc_id, int(h.page_num)) for h in res.hits]

# Visual-fused ranked page identities = gated weighted RRF(visual, text).
fused_ranked_by_query = {}
for q in gold_queries:
    qid = q["query_id"]
    fused = rrf_fuse(
        visual_ranked_by_query.get(qid, []),
        text_ranked_by_query.get(qid, []),
        k=60,
        text_weight=4.0,
        visual_weight=1.0,
        page_text_lengths=PAGE_TEXT_LENGTHS,
        empty_text_boost=3.0,
    )
    fused_ranked_by_query[qid] = [page_key for page_key, _score in fused]

assert_fused_recall_not_below_text(gold_targets, text_ranked_by_query, fused_ranked_by_query, k_values=K_VALUES)

def _eval_mode(ranked_by_query):
    out = {}
    for k in K_VALUES:
        retrieved = {qid: [(d, p, 1.0) for d, p in ranked[:k]] for qid, ranked in ranked_by_query.items()}
        recall = compute_retrieval_recall_at_k(gold_targets, retrieved, k=k)
        cited = {qid: ranked[:k] for qid, ranked in ranked_by_query.items()}
        citation = compute_page_level_citation_accuracy(gold_targets, cited)
        out[f"recall@{k}"] = recall.macro_recall
        out[f"citation_acc@{k}"] = float(citation["macro_accuracy"])
    return out

text_metrics = _eval_mode(text_ranked_by_query)
fused_metrics = _eval_mode(fused_ranked_by_query)

metric_keys = [f"recall@{k}" for k in K_VALUES] + [f"citation_acc@{k}" for k in K_VALUES]
print(f"{'metric':<18}{'text-only':>12}{'visual-fused':>14}{'delta':>10}")
print("-" * 54)
for key in metric_keys:
    t = text_metrics[key]; f = fused_metrics[key]
    print(f"{key:<18}{t:>12.3f}{f:>14.3f}{(f - t):>+10.3f}")
print("\n>>> Acceptance passed: fused recall@5/@10 is not below text-only.")

## 8. Example-3 proof: the four `rq_ex3_*` image-only gold pages appear in visual top-k

Doc `5543408c4dacc48b`, gold page **2 (0-indexed)** is scanned image-only — text recall@5 is
structurally 0 on all four `rq_ex3_*` queries (the text indexer excludes empty-text pages).
This cell prints, for each `rq_ex3_*` query, whether each gold page appears in the **visual**
and **fused** top-k — the core proof that the visual tier makes image-only pages retrievable.

In [ ]:
rq_ex3_ids = [q["query_id"] for q in gold_queries if str(q["query_id"]).startswith("rq_ex3")]
print(f"rq_ex3 queries found: {rq_ex3_ids}\n")

PROOF_K = 5
RQ_EX3_VISUAL_RANK_THRESHOLD = 10
rq_ex3_rank_failures = []

def _rank_of(ranked, target):
    try:
        return list(ranked).index(target) + 1
    except ValueError:
        return None

for qid in rq_ex3_ids:
    targets = set(gold_targets.get(qid, []))
    visual_ranked = visual_ranked_by_query.get(qid, [])
    visual_topk = set(visual_ranked[:PROOF_K])
    fused_topk = set(fused_ranked_by_query.get(qid, [])[:PROOF_K])
    text_topk = set(text_ranked_by_query.get(qid, [])[:PROOF_K])
    print(f"{qid}: gold={sorted(targets)}")
    for tgt in sorted(targets):
        visual_rank = _rank_of(visual_ranked, tgt)
        print(
            f"    page {tgt}: text@{PROOF_K}={'HIT' if tgt in text_topk else 'miss':<4} "
            f"visual@{PROOF_K}={'HIT' if tgt in visual_topk else 'miss':<4} "
            f"fused@{PROOF_K}={'HIT' if tgt in fused_topk else 'miss':<4} "
            f"visual_rank={visual_rank}"
        )
        if visual_rank is None or visual_rank > RQ_EX3_VISUAL_RANK_THRESHOLD:
            rq_ex3_rank_failures.append({"query_id": qid, "target": tgt, "visual_rank": visual_rank})
print("\n>>> Acceptance: every rq_ex3 target must rank <=", RQ_EX3_VISUAL_RANK_THRESHOLD, "visually.")
assert not rq_ex3_rank_failures, f"rq_ex3 visual rank acceptance failed: {rq_ex3_rank_failures}"

## 9. Export the run report and index artifact

The report is the durable diagnostic artifact for this GPU run: versions, load-report summary,
point count, text/visual/fused target ranks, top-10 visual hits with scores, and the `rq_ex3`
acceptance threshold. Download it with the Qdrant artifact; do not commit `compliance.db` or Qdrant storage.

In [ ]:
import json, shutil
from pathlib import Path


def _target_ranks(ranked_by_query):
    out = {}
    for qid, targets in gold_targets.items():
        ranked = ranked_by_query.get(qid, [])
        out[qid] = []
        for target in targets:
            out[qid].append({"doc_id": target[0], "page_num": int(target[1]), "rank": _rank_of(ranked, target)})
    return out

per_query_target_ranks = {
    "text": _target_ranks(text_ranked_by_query),
    "visual": _target_ranks(visual_ranked_by_query),
    "fused": _target_ranks(fused_ranked_by_query),
}
report = {
    "collection_name": name,
    "run_id": visual_run.run_id,
    "point_count": int(client.count(name).count),
    "model_version": "vidore/colqwen2.5-v0.2",
    "versions": VERSION_TRIPLE,
    "load_report": LOAD_REPORT,
    "metrics": {"text_only": text_metrics, "visual_fused": fused_metrics},
    "per_query_target_ranks": per_query_target_ranks,
    "top10_visual_hits": {qid: hits[:10] for qid, hits in visual_hits_by_query.items()},
    "rq_ex3_visual_rank_threshold": RQ_EX3_VISUAL_RANK_THRESHOLD,
    "rq_ex3_rank_failures": rq_ex3_rank_failures,
}
report_path = Path("/content/visual_retrieval_run_report.json")
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

md_lines = [
    "# Visual Retrieval Run Report",
    "",
    f"- collection: `{name}`",
    f"- run_id: `{visual_run.run_id}`",
    f"- point_count: `{int(client.count(name).count)}`",
    f"- versions: `{json.dumps(VERSION_TRIPLE, sort_keys=True)}`",
    f"- load_report_missing_count: `{LOAD_REPORT['missing_count']}`",
    f"- load_report_unexpected_count: `{LOAD_REPORT['unexpected_count']}`",
    f"- rq_ex3_visual_rank_threshold: `{RQ_EX3_VISUAL_RANK_THRESHOLD}`",
    f"- rq_ex3_rank_failures: `{json.dumps(rq_ex3_rank_failures)}`",
    "",
    "## Metrics",
    "",
]
for mode_name, metrics in (("text_only", text_metrics), ("visual_fused", fused_metrics)):
    md_lines.append(f"### {mode_name}")
    for key, value in metrics.items():
        md_lines.append(f"- {key}: {value:.3f}")
    md_lines.append("")
Path("/content/visual_retrieval_run_report.md").write_text("\n".join(md_lines), encoding="utf-8")

manifest = {
    "collection_name": name,
    "run_id": visual_run.run_id,
    "point_count": int(client.count(name).count),
    "model_version": "vidore/colqwen2.5-v0.2",
    **VERSION_TRIPLE,
    "report_json": str(report_path),
}
with open("/content/visual_index_manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
shutil.make_archive("/content/qdrant_storage_artifact", "zip", QDRANT_PATH)
print("manifest:", json.dumps(manifest, indent=2))
print("run report: /content/visual_retrieval_run_report.json and /content/visual_retrieval_run_report.md")
print("artifact: /content/qdrant_storage_artifact.zip (download for the deferred demo; do NOT commit)")